# Step 1 - Prepare the data using pytorch

In [133]:
import torch

In [134]:
X = torch.tensor([
    [1000.0],
    [1500.0],
    [2000.0],
    [2500.0],
    [3000.0],
    [3500.0],
    [4000.0],
    [4500.0],
    [5000.0],
    [5500.0]
])

In [135]:
y = torch.tensor([
    [100000.0],
    [150000.0],
    [200000.0],
    [250000.0],
    [300000.0],
    [350000.0],
    [400000.0],
    [450000.0],
    [500000.0],
    [550000.0]
])

In [136]:
print(X.shape)      # 10, 1 --> 10 rows (10 houses), 1 column (house size)
print(y.shape)

torch.Size([10, 1])
torch.Size([10, 1])


In [137]:
# Inspect our data
print(f"Features (X): {X}")
print(f"Targets (y): {y}")
print(f"X Shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"X Data type: {X.dtype}")
print(f"y data type: {y.dtype}")

Features (X): tensor([[1000.],
        [1500.],
        [2000.],
        [2500.],
        [3000.],
        [3500.],
        [4000.],
        [4500.],
        [5000.],
        [5500.]])
Targets (y): tensor([[100000.],
        [150000.],
        [200000.],
        [250000.],
        [300000.],
        [350000.],
        [400000.],
        [450000.],
        [500000.],
        [550000.]])
X Shape: torch.Size([10, 1])
y shape: torch.Size([10, 1])
X Data type: torch.float32
y data type: torch.float32


# Step 2 - Splitting data into Training and Validation sets

In [138]:
from sklearn.model_selection import train_test_split

In [139]:
# X - Input feature
# y - Target values
# test_size = 0.2 --> 20% allocation for the validation set
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [140]:
print(f"X train shape: {X_train.shape}")
print(f"X val shape: {X_val.shape}")

print(f"y train shape: {y_train.shape}")
print(f"y val shape: {y_val.shape}")

X train shape: torch.Size([8, 1])
X val shape: torch.Size([2, 1])
y train shape: torch.Size([8, 1])
y val shape: torch.Size([2, 1])


# Step 3 - Creating TensorDataset and DataLoader

In [141]:
# TensorDataset --> It allows us to combine multiple tensors into a single dataset. In our case, we want to combine X_train (house size) and y_train (house price)
from torch.utils.data import TensorDataset

In [142]:
# Now create our validation and training datasets
train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_val, y_val)

In [143]:
print(train_dataset[0])
print(validation_dataset[0])

(tensor([3500.]), tensor([350000.]))
(tensor([5000.]), tensor([500000.]))


In [144]:
# DataLoader divides the dataset into smaller groups called batches
from torch.utils.data import DataLoader

In [145]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# train_dataset --> Training Dataset from which the samples are loaded
# batch_size --> Two samples per batch
# shuffle = True --> Randomize training sample order because it prevents the model from repeatedly seeing the training samples in the same order across epochs

In [146]:
len(train_loader)

4

In [147]:
validation_loader = DataLoader(validation_dataset, batch_size=2, shuffle=False)
# We generally keep the validation data in a fixed order for consistent inspection and evaluation

In [148]:
print(len(validation_loader))

1


### How does DataLoader work?

In [149]:
# Our training dataset contains 8 samples and our batch size = 2. So we will get four batches and their corresponding prices

for X_batch, y_batch in train_loader:
    print(f"X_batch: {X_batch}")
    print("-----------")
    print(f"y_batch: {y_batch}")
    print("-----------")

X_batch: tensor([[2500.],
        [2000.]])
-----------
y_batch: tensor([[250000.],
        [200000.]])
-----------
X_batch: tensor([[1000.],
        [5500.]])
-----------
y_batch: tensor([[100000.],
        [550000.]])
-----------
X_batch: tensor([[4000.],
        [3500.]])
-----------
y_batch: tensor([[400000.],
        [350000.]])
-----------
X_batch: tensor([[4500.],
        [3000.]])
-----------
y_batch: tensor([[450000.],
        [300000.]])
-----------


# Step 4 - Inspect our training and validation data

In [150]:
# Inspect the dataset size
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")

Training samples: 8
Validation samples: 2


# Step 5 - Inspecting a Training Batch

In [151]:
# The DataLoader automatically generates X_batch, y_batch during the iteration in the loop above

X_batch, y_batch = next(iter(train_loader))
# iter(train_loader) creates an iterator over our DataLoader.
# next() retrieves the next available batch.
# We unpack that batch into X_batch and y_batch

In [152]:
train_loader

In [153]:
print("X_batch:")
print(X_batch)

print("\ny_batch:")
print(y_batch)

X_batch:
tensor([[3500.],
        [2000.]])

y_batch:
tensor([[350000.],
        [200000.]])


# Step 6 - Defining our Neural network Model

In [154]:
# nn provides ready-made components for building neural networks, including linear layers, activation functions, loss functions, other neural network building blocks
import torch.nn as nn

In [155]:
model = nn.Sequential(nn.Linear(1, 1))

# nn.Sequential -- Lets us arrange neural network layers in a sequence
# nn.Linear --> Performs our linear regression calculation. The output of one layer is passed to next layer automatically

In [156]:
print(model)

# Sequential --> Our model is a sequence of layers
# (0) --> Index of the first layer
# Linear --> This layer performs a linear transformation
# in_features = 1 -> Accepts one input feature
# out_features = 1 -> Produces one output
# bias = True -> The layer includes a bias parameter

Sequential(
  (0): Linear(in_features=1, out_features=1, bias=True)
)


In [157]:
# Our model needs to learn two parameters: weights (w), and bias (b). Pytorch stores these parameters internally. model.names_parameters() allows us to retrieve the model's trainable parameters along with their names
for name, param in model.named_parameters():
    print(name, param)

# 0.weight / 0.bias --> Because our model is nn.sequential. Our first layer has index 0. Therefore 0.weight is the weight of layer 0. and 0.bias is the bias of layer 0

0.weight Parameter containing:
tensor([[0.5961]], requires_grad=True)
0.bias Parameter containing:
tensor([0.3241], requires_grad=True)


In [158]:
# generate Predictions
predictions = model(X_batch)
print(predictions)
print(predictions.shape)

tensor([[2086.7197],
        [1192.5503]], grad_fn=<AddmmBackward0>)
torch.Size([2, 1])


# Step 7 - Choosing the loss function

In [159]:
# Define a loss function
loss_function = nn.MSELoss()

In [160]:
# Calculate loss
loss_value = loss_function(predictions, y_batch)

In [161]:
# Predicted values
print(predictions)

tensor([[2086.7197],
        [1192.5503]], grad_fn=<AddmmBackward0>)


In [162]:
# Actual values
print(y_batch)

tensor([[350000.],
        [200000.]])


In [163]:
# The loss value between the predicted and the actual
print(loss_value)

tensor(8.0284e+10, grad_fn=<MseLossBackward0>)


# Step 8 - Choosing the Optimizer

In [164]:
import torch.optim as optim

In [165]:
# Define the optimizer
optimizer = optim.SGD(model.parameters(), lr=0.01)

# model.parameters () --> The parameters that the optimizer should update

In [167]:
print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)
